# Silver Layer: Data Integration and Null Handling

This notebook performs data integration (joining Retention and BOB datasets). It ensures the output contains only unique account numbers by grouping by account number. During the grouping:
- Total, Lost, and Active combinations are tracked to categorize **Churn**.
- All other numerical features are averaged (`mean`).
- All other categorical features use the most frequent value (`mode`).

In [23]:
from importlib import reload
import sys
from pathlib import Path
BASE_DIR = Path().resolve().parents[1]
sys.path.append(str(BASE_DIR))
print("Project Root:", BASE_DIR)

Project Root: C:\Users\sjram\OneDrive\Documents\customer-churn-analysis


In [24]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
from src.features.churn_logic import create_churn_labels
import src.features.churn_logic as churn_logic
from src.features.aggregation import aggregate_bob_to_customer
reload(churn_logic)

<module 'src.features.churn_logic' from 'C:\\Users\\sjram\\OneDrive\\Documents\\customer-churn-analysis\\src\\features\\churn_logic.py'>

In [25]:
INTERIM_DATA = BASE_DIR / "data" / "02_intermediate"

bob = pd.read_parquet(INTERIM_DATA / "bob_clean.parquet")
retention = pd.read_parquet(INTERIM_DATA / "retention_clean.parquet")

In [26]:
retention, customer_churn_df = create_churn_labels(retention)
customer_churn_df['customer_churn'].value_counts()

customer_churn
0    9239
2    7405
1    4882
Name: count, dtype: int64

In [27]:
bob_agg = aggregate_bob_to_customer(bob)

print(bob_agg.shape)
bob_agg.head()

(23772, 27)


,account_number,product_bob,fee_bob,total_revenue,service_interval,avg_unit_amount,billing_interval,company_sizing,postal_code,branch,...,system_status,is_bob,bpg,msdyn_product_number,product_name,billing_period,machine,machine_variant,chemistry,total_agreements
0,GBA221737,8680.100,0.00,8680.10,NaN,NaN,NaN,Unknown,NR28 0TY,Elmswell,...,Active,Yes,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,1
1,UK02-000167723,1808.375,0.00,3616.75,NaN,NaN,NaN,Unknown,ME8 6PS,Maidstone,...,Active,Yes,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,2
2,UK02-000167746,2840.030,159.96,2999.99,6.0,236.67,1.0,Unknown,CF35 5AQ,Cardiff,...,Active,Yes,04. Manual Aqueous Cleaner,IM100BS2-000562006,M100 BS2 + 60L of KLEEN1000M + Waste Aqueous S...,monthly,M100,S2,KLEEN1000M,1
3,UK02-000167747,2540.140,0.00,2540.14,NaN,NaN,NaN,Unknown,SK4 1BJ,Chester,...,Active,Yes,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,1
4,UK02-000167753,4334.270,0.00,4334.27,NaN,NaN,NaN,Unknown,EX39 4LQ,Exeter,...,Active,Yes,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,1


In [28]:
customer_churn_df = customer_churn_df.merge(
    bob_agg,
    left_on='customer_account_number',
    right_on='account_number',
    how='left'
)

In [29]:
customer_churn_df.isnull().sum().sort_values(ascending=False)

billing_interval           16225
service_interval           15939
avg_unit_amount            15922
product_bob                15898
total_revenue              15898
fee_bob                    15898
account_number             15898
company_sizing             15898
branch                     15898
postal_code                15898
vat_number                 15898
agreement_type             15898
renewal_type               15898
agreement_end_date         15898
agreement_start_date       15898
agreement_number           15898
bpg                        15898
msdyn_product_number       15898
product_name               15898
billing_period             15898
machine                    15898
line_of_business           15898
system_status              15898
is_bob                     15898
chemistry                  15898
machine_variant            15898
total_agreements_y         15898
active_agreements              0
customer_churn                 0
lost_agreements                0
total_agre

In [30]:
customer_churn_df['customer_account_number'].nunique()

21526

In [31]:
for col in customer_churn_df.columns:

    if customer_churn_df[col].dtype == 'object':
        customer_churn_df[col] = customer_churn_df[col].fillna("Unknown")

    else:
        customer_churn_df[col] = customer_churn_df[col].fillna(0)

In [32]:
customer_churn_df = customer_churn_df.drop(columns=['total_agreements_x'], errors='ignore')

customer_churn_df = customer_churn_df.rename(columns={
    'total_agreements_y': 'total_agreements'
})

In [33]:
customer_churn_df.columns

Index(['customer_account_number', 'lost_agreements', 'active_agreements',
       'customer_churn', 'account_number', 'product_bob', 'fee_bob',
       'total_revenue', 'service_interval', 'avg_unit_amount',
       'billing_interval', 'company_sizing', 'postal_code', 'branch',
       'vat_number', 'agreement_number', 'agreement_start_date',
       'agreement_end_date', 'renewal_type', 'agreement_type',
       'line_of_business', 'system_status', 'is_bob', 'bpg',
       'msdyn_product_number', 'product_name', 'billing_period', 'machine',
       'machine_variant', 'chemistry', 'total_agreements'],
      dtype='object')

In [34]:
customer_churn_df['revenue_per_agreement'] = customer_churn_df['total_revenue'] / (customer_churn_df['total_agreements'] + 1)

customer_churn_df['service_intensity'] = customer_churn_df['service_interval'] / (customer_churn_df['total_agreements'] + 1)

In [35]:
customer_churn_df.isnull().sum()

customer_account_number    0
lost_agreements            0
active_agreements          0
customer_churn             0
account_number             0
product_bob                0
fee_bob                    0
total_revenue              0
service_interval           0
avg_unit_amount            0
billing_interval           0
company_sizing             0
postal_code                0
branch                     0
vat_number                 0
agreement_number           0
agreement_start_date       0
agreement_end_date         0
renewal_type               0
agreement_type             0
line_of_business           0
system_status              0
is_bob                     0
bpg                        0
msdyn_product_number       0
product_name               0
billing_period             0
machine                    0
machine_variant            0
chemistry                  0
total_agreements           0
revenue_per_agreement      0
service_intensity          0
dtype: int64

In [36]:
for col in ['total_revenue', 'avg_unit_amount']:

    Q1 = customer_churn_df[col].quantile(0.25)
    Q3 = customer_churn_df[col].quantile(0.75)
    IQR = Q3 - Q1

    customer_churn_df[col] = customer_churn_df[col].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)

In [37]:
customer_churn_df['total_revenue'] = np.log1p(customer_churn_df['total_revenue'])
customer_churn_df['avg_unit_amount'] = np.log1p(customer_churn_df['avg_unit_amount'])

In [38]:
PROCESSED_DATA = BASE_DIR / "data" / "03_processed"
PROCESSED_DATA.mkdir(parents=True, exist_ok=True)

customer_churn_df.to_parquet(PROCESSED_DATA / "final_dataset.parquet", index=False)